# NFL Game Outcome Prediction (XGBoost)

Predict home win probability, projected spread, and projected total for any NFL game
using team-level rolling aggregates from `nflfastR` and an XGBoost ensemble.

**What's inside:**
- `NFLGamePredictor` — wraps an XGBoost classifier (win probability) plus two
  XGBoost regressors (spread, total).
- `NFLGameFeatures` — 14 hand-engineered team-strength features.
- `train_default_model` — synthetic-data bootstrap so you can run the full
  pipeline without network access. Plug in real `nflfastR` play-by-play to
  retrain for production use.


In [1]:
from sportsquant.models.predictive.nfl_game_model import (
    NFLGameFeatures,
    NFLGamePredictor,
    feature_columns,
    train_default_model,
)
from sportsquant.data.nfl import NFLDataConfig, NFLDataPipeline
from sportsquant.models.predictive.nfl_game_model import build_features_from_pipeline

print('NFL XGBoost predictor ready')
print('Feature columns:', feature_columns())

NFL XGBoost predictor ready
Feature columns: ['home_ppg_for', 'home_ppg_against', 'home_yards_per_play', 'home_turnover_rate', 'home_qb_rating', 'away_ppg_for', 'away_ppg_against', 'away_yards_per_play', 'away_turnover_rate', 'away_qb_rating', 'ppg_differential', 'defense_differential', 'home_advantage', 'rest_advantage']


## 1. Train a default model

We bootstrap with a synthetic dataset so the notebook runs offline. For real
production use, swap in `nflfastR` play-by-play + nflverse schedules and call
`predictor.train(real_df)` directly.

In [2]:
predictor = train_default_model(n_games=600, verbose=True)
print('Trained. Feature importances:')
for feat, imp in sorted(predictor.feature_importances.items(), key=lambda x: -x[1]):
    print(f'  {feat:24s} {imp:.4f}')

Trained. Feature importances:
  away_qb_rating           0.1623
  home_qb_rating           0.1357
  defense_differential     0.1250
  ppg_differential         0.1243
  home_ppg_against         0.0723
  home_ppg_for             0.0652
  away_ppg_against         0.0643
  away_ppg_for             0.0580
  home_turnover_rate       0.0429
  away_turnover_rate       0.0423
  away_yards_per_play      0.0369
  home_yards_per_play      0.0366
  rest_advantage           0.0343
  home_advantage           0.0000


## 2. Build features for a real game

Pulls rolling team aggregates from nflfastR via the `NFLDataPipeline`. When
network/cache is unavailable the function gracefully returns zero-valued
features so downstream code still runs.

In [3]:
pipeline = NFLDataPipeline(config=NFLDataConfig())
features = build_features_from_pipeline(
    pipeline,
    home_team="KC",
    away_team="BAL",
    season=2024,
    week=10,
)
print('Home team:', features.home_team)
print('Away team:', features.away_team)
print('Home PPG for:', round(features.home_ppg_for, 2))
print('Away PPG for:', round(features.away_ppg_for, 2))
print('PPG differential:', round(features.ppg_differential, 2))
print('Defense differential:', round(features.defense_differential, 2))

Home team: KC
Away team: BAL
Home PPG for: 43.07
Away PPG for: 50.87
PPG differential: -7.79
Defense differential: -7.79


## 3. Predict the outcome

In [4]:
prediction = predictor.predict(features)
print(f'{prediction.away_team} @ {prediction.home_team}')
print(f'  Home win prob: {prediction.home_win_prob:.1%}')
print(f'  Away win prob: {prediction.away_win_prob:.1%}')
print(f'  Projected spread: {prediction.proj_spread:+.1f}')
print(f'  Projected total: {prediction.proj_total:.1f}')

BAL @ KC
  Home win prob: 69.6%
  Away win prob: 30.4%
  Projected spread: +3.8
  Projected total: 83.5


## 4. Scenario analysis

Compare win probability across multiple hypothetical matchups to find edges
versus the market line.

In [5]:
matchups = [
    ('KC', 'BAL', 28.0, 22.0, 100.0, 85.0),
    ('KC', 'CAR', 28.0, 15.0, 100.0, 70.0),
    ('KC', 'BUF', 25.0, 25.0, 95.0, 95.0),
    ('CAR', 'KC', 15.0, 28.0, 70.0, 100.0),
]
for home, away, hppg, appg, hqbr, aqbr in matchups:
    f = NFLGameFeatures(
        home_team=home, away_team=away, season=2024, week=10,
        home_ppg_for=hppg, away_ppg_for=appg,
        home_qb_rating=hqbr, away_qb_rating=aqbr,
        home_ppg_against=appg, away_ppg_against=hppg,
    )
    p = predictor.predict(f)
    print(f'  {away:3s} @ {home:3s}  prob(home)={p.home_win_prob:5.1%}  spread={p.proj_spread:+5.1f}  total={p.proj_total:5.1f}')

  BAL @ KC   prob(home)=97.6%  spread=+13.9  total= 96.7
  CAR @ KC   prob(home)=99.0%  spread=+15.0  total= 93.7
  BUF @ KC   prob(home)=73.5%  spread= +0.1  total= 93.7
  KC  @ CAR  prob(home)= 1.3%  spread= -4.4  total= 92.7


## 5. Save the trained model

Saved models can be loaded later via `NFLGamePredictor.load(path)` — useful
for production scoring jobs that retrain weekly.

In [6]:
from pathlib import Path
save_path = Path('notebooks/football/cache/nfl_game_model')
save_path.mkdir(parents=True, exist_ok=True)
predictor.save(save_path)
print(f'Saved to {save_path}')
print('Files:', sorted(p.name for p in save_path.iterdir()))

# Round-trip load to confirm
reloaded = NFLGamePredictor.load(save_path)
print('Reloaded home win prob:', reloaded.predict(features).home_win_prob)

Saved to notebooks/football/cache/nfl_game_model
Files: ['classifier.json', 'meta.json', 'spread.json', 'total.json']
Reloaded home win prob: 0.6962815523147583


## Summary

- The `NFLGamePredictor` exposes three XGBoost sub-models trained jointly on the
  same 14-feature schema.
- Synthetic bootstrap lets you iterate without nflfastR access; production
  training swaps `_generate_synthetic_dataset()` for a real `nflfastR` join
  with nflverse schedules.
- Feature importances surface the model signal at a glance: QB rating, PPG
  differential, and defense differential dominate.
- `NFLDataPipeline.build_features_from_pipeline` is the canonical feature
  builder; it gracefully degrades to zeroed features when network/cache is
  unavailable.

Next steps for production:
1. Replace `_generate_synthetic_dataset()` with real play-by-play joins.
2. Calibrate win probabilities (Platt scaling or isotonic regression).
3. Wire into the `nfl predict-game` CLI for ad-hoc usage.
